# 연고관광 전체 원변수·파생변수 클러스터 lift 분석

## 분석 목적과 범위

2023~2025년 국민여행조사의 관광·휴양 활동을 포함한 가족·친지·친구 방문
여행(CASE 2) 중 공식 최종 사전층화 군집 결과에서, 각 군집이 같은 여행층
평균과 비교해 어떤 원변수·파생변수 수준에서 과대표 또는 과소대표되는지
확인한다.

- 분석 단위는 `YEAR + ID`로 식별된 CASE 2 여행 사례이며, 입력 라벨링
  자료의 한 행을 한 사례로 사용한다.
- `WT_DOM`으로 가중 평균·비율을 계산한다. 가중 합은 연간 고유 방문자 수가
  아니라 가중 여행 사례 수다.
- 군집은 다시 학습하지 않으며, 공식 `baseline_stratified_cluster_labeled_data`
  의 `층`, `군집` 라벨을 그대로 사용한다.
- 기존 33개 수치형 lift는 같은 정의로 재현 검산한다. 범주형 결과는 수준별
  가중 비율을 사용하며, 연관성을 인과효과로 해석하지 않는다.

In [ ]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").is_dir():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("src 폴더가 있는 프로젝트 루트를 찾지 못했습니다.")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

from src.path import (
    NATIONAL_TRAVEL_SURVEY_PREPROCESSED_DATA_DIR,
    OUTPUTS_DIR,
    create_output_directories,
)
from src.visualization import COLORS, apply_plot_style

apply_plot_style()

NOTEBOOK_NAME = "260726_연고관광_전체변수_클러스터_lift.ipynb"
FIGURES_DIR, TABLES_DIR = create_output_directories(NOTEBOOK_NAME)
SOURCE_DIR = OUTPUTS_DIR / "260720_연고관광_사전층화_클러스터링_3개년_v4"
INPUT_PATH = (
    SOURCE_DIR
    / "tables"
    / "baseline_stratified_cluster_labeled_data.csv"
)
PRIOR_LIFT_PATH = (
    SOURCE_DIR / "tables" / "baseline_stratified_cluster_lift.csv"
)
AUTOENCODER_DIR = SOURCE_DIR / "model" / "baseline" / "autoencoder"
CODEBOOK_PATH = (
    NATIONAL_TRAVEL_SURVEY_PREPROCESSED_DATA_DIR
    / "national_travel_survey_2023_2025_preprocessed_codebook.csv"
)
DISPLAY_NAME_PATH = (
    NATIONAL_TRAVEL_SURVEY_PREPROCESSED_DATA_DIR
    / "national_travel_survey_2023_2025_display_name.csv"
)

ID_COLUMN = "ID"
WEIGHT_COLUMN = "WT_DOM"
STRATUM_COLUMN = "층"
CLUSTER_COLUMN = "군집"
EXPECTED_CLUSTER_COUNTS = {"당일여행": 2, "숙박여행": 3}
MIN_CLUSTER_N = 30
MIN_BASE_SHARE = 0.01
TOP_FEATURES_PER_CLUSTER = 12
TOLERANCE = 1e-9

# 대표 히트맵은 이동 방식·교통비를 각각 하나의 지표로만 보여 준다.
# 순위 문항은 1순위 원변수만 사용하고, 가중점수와 2·3순위는 제외한다.
# 전체 lift 결과표에는 아래 변수를 모두 보존한다.
VISUAL_RANKED_PREFIXES = ("A3", "A5", "A5A", "D_TRA_B7", "ZQ1")
VISUAL_EXCLUDED_VARIABLES = {
    "primary_transport",
    "car_dependence",
    "public_transport",
    *{
        f"{prefix}_{rank}"
        for prefix in VISUAL_RANKED_PREFIXES
        for rank in range(2, 4)
    },
}
VISUAL_EXCLUDED_PREFIXES = ("NA8F",)

# 해석 목적과 맞지 않거나 주관식 확인이 필요한 원변수는 lift 대상에서 제외한다.
INTERPRETATION_EXCLUSIONS = {
    "BARA": "거주지역은 해석 범위에서 제외",
    **{
        f"D_TRA_{spot_no}_SPOT": "방문지 정보는 해석 범위에서 제외"
        for spot_no in range(1, 13)
    },
    "A6": "party_size와 같은 일행 수 원변수라 제외",
    "D_TRA_COST": "1인당 지출 지표를 사용하므로 여행 총경비는 제외",
    "D_TRA_ONE_COST": "spend_per_person와 같은 1인당 지출 원변수라 제외",
    "destination_spend": "총 지출액은 1인당·1일당 지출 지표와 중복돼 제외",
    "log_spend_per_person_day": "spend_per_person_day의 로그 변환값이라 제외",
    "A4_21": "기타 활동은 주관식 세부 확인이 필요해 제외",
    "A7_10": "기타 소비 항목은 주관식 세부 확인이 필요해 제외",
    "NA7_10": "기타 소비 항목은 주관식 세부 확인이 필요해 제외",
    "A8C_14": "기타 소비 항목은 주관식 세부 확인이 필요해 제외",
    "NA8C_14": "기타 소비 항목은 주관식 세부 확인이 필요해 제외",
    "A8E_4": "기타 소비 항목은 주관식 세부 확인이 필요해 제외",
    "NA8E_4": "기타 소비 항목은 주관식 세부 확인이 필요해 제외",
    "A8F_6": "기타 소비 항목은 주관식 세부 확인이 필요해 제외",
    "NA8F_6": "기타 소비 항목은 주관식 세부 확인이 필요해 제외",
    "A8G_6": "기타 소비 항목은 주관식 세부 확인이 필요해 제외",
    "NA8G_6": "기타 소비 항목은 주관식 세부 확인이 필요해 제외",
    "A8I_3": "기타 소비 항목은 주관식 세부 확인이 필요해 제외",
    "NA8I_3": "기타 소비 항목은 주관식 세부 확인이 필요해 제외",
    "A8J": "기타 소비 항목은 주관식 세부 확인이 필요해 제외",
    "A8J_1": "기타 소비 항목은 주관식 세부 확인이 필요해 제외",
    "NA8J_1": "기타 소비 항목은 주관식 세부 확인이 필요해 제외",
    "A8SC_7": "기타 소비 비중은 주관식 세부 확인이 필요해 제외",
    "A8SD_6": "기타 소비 비중은 주관식 세부 확인이 필요해 제외",
    "A8SE_3": "기타 소비 비중은 주관식 세부 확인이 필요해 제외",
    "A8SF_4": "기타 소비 비중은 주관식 세부 확인이 필요해 제외",
}
INTERPRETATION_EXCLUDED_LEVELS = {
    "A4A": {"21"},
    **{
        f"A9A_{question_no}": {"9"}
        for question_no in range(1, 13)
    },
}

DERIVED_DISPLAY_NAMES = {
    "nights": "숙박일수",
    "overnight_trip": "숙박여행 여부",
    "package_used": "여행상품 이용",
    "reservation_any": "사전예약 여부",
    "reservation_count": "사전예약 항목 수",
    "information_none": "정보 없이 방문",
    "information_online": "온라인 정보 이용",
    "information_people": "주변인 정보 이용",
    "activity_nature": "자연·경관 활동",
    "activity_food": "음식관광 활동",
    "activity_culture": "문화·예술 활동",
    "activity_experience": "체험 활동",
    "activity_leisure": "레저·오락 활동",
    "activity_wellness": "휴식·휴양 활동",
    "activity_shopping": "쇼핑 활동",
    "activity_event": "축제·이벤트 활동",
    "activity_group_count": "활동 대분류 수",
    "party_size": "동행 인원",
    "party_group": "동행 규모",
    "with_child": "미성년자 동반",
    "visit_region_count": "방문 시군구 수",
    "multi_region": "다지역 방문",
    "family_home_lodging": "가족·친지집 숙박",
    "commercial_lodging": "상업숙박시설 이용",
    "primary_transport": "주 이용 교통수단",
    "car_dependence": "자가용·렌터카 이용",
    "public_transport": "대중교통 이용",
    "trip_days": "여행일수",
    "spend_per_person": "1인 여행경비",
    "spend_per_person_day": "1인·1일 여행경비",
    "log_spend_per_person_day": "1인·1일 여행경비(로그)",
    "destination_spend": "여행지 총지출",
    "share_lodging": "숙박비 비중",
    "share_food": "식음비 비중",
    "share_transport": "교통비 비중",
    "share_activity": "활동비 비중",
    "share_shopping": "쇼핑비 비중",
    "activity_spend": "활동비 지출 여부",
    "local_food_shopping_spend": "지역 식음·쇼핑 지출",
}

# 파생변수 39개의 의미 유형을 명시적으로 관리한다.
DERIVED_TYPES = {
    "nights": "연속·계수형",
    "overnight_trip": "이진형",
    "package_used": "이진형",
    "reservation_any": "이진형",
    "reservation_count": "연속·계수형",
    "information_none": "이진형",
    "information_online": "이진형",
    "information_people": "이진형",
    "activity_nature": "이진형",
    "activity_food": "이진형",
    "activity_culture": "이진형",
    "activity_experience": "이진형",
    "activity_leisure": "이진형",
    "activity_wellness": "이진형",
    "activity_shopping": "이진형",
    "activity_event": "이진형",
    "activity_group_count": "연속·계수형",
    "party_size": "연속·계수형",
    "party_group": "범주형",
    "with_child": "이진형",
    "visit_region_count": "연속·계수형",
    "multi_region": "이진형",
    "family_home_lodging": "이진형",
    "commercial_lodging": "이진형",
    "primary_transport": "범주형",
    "car_dependence": "이진형",
    "public_transport": "이진형",
    "trip_days": "연속·계수형",
    "spend_per_person": "연속·계수형",
    "spend_per_person_day": "연속·계수형",
    "log_spend_per_person_day": "연속·계수형",
    "destination_spend": "연속·계수형",
    "share_lodging": "연속·계수형",
    "share_food": "연속·계수형",
    "share_transport": "연속·계수형",
    "share_activity": "연속·계수형",
    "share_shopping": "연속·계수형",
    "activity_spend": "이진형",
    "local_food_shopping_spend": "이진형",
}

print(f"입력 자료: {INPUT_PATH}")
print(f"결과표 디렉터리: {TABLES_DIR}")
print(f"그림 디렉터리: {FIGURES_DIR}")

## 데이터와 기존 오토인코더 캐시 확인

코드북의 `nominal` 또는 값 라벨이 있는 코드형 변수는 범주형으로, 값 라벨이
없는 `scale` 변수는 연속·계수형으로 취급한다. 숫자로 저장된 지역·응답 코드는
평균을 계산하지 않는다. 파생변수는 위 사전에 따라 이진형·범주형·연속·계수형을
구분한다.

오토인코더 캐시는 16차원 임베딩 배열과 검증 R²만 보관하는지 확인한다. 디코더가
저장되지 않았으므로 새 재구성 성능은 계산하지 않는다.

In [ ]:
data = pd.read_csv(INPUT_PATH, low_memory=False)
codebook = pd.read_csv(CODEBOOK_PATH, encoding="utf-8-sig")
display_names = pd.read_csv(DISPLAY_NAME_PATH, encoding="utf-8-sig")
raw_feature_columns = set(data.columns) - set(DERIVED_TYPES) - {
    ID_COLUMN,
    WEIGHT_COLUMN,
    STRATUM_COLUMN,
    CLUSTER_COLUMN,
}
missing_display_names = raw_feature_columns - set(display_names["column_name"])
assert not missing_display_names, (
    f"표시명이 없는 원변수: {sorted(missing_display_names)}"
)

assert len(data) == 12_790, f"예상 행 수와 다릅니다: {len(data):,}"
assert data[ID_COLUMN].notna().all()
assert data[ID_COLUMN].is_unique
weights = pd.to_numeric(data[WEIGHT_COLUMN], errors="coerce")
assert weights.notna().all()
assert weights.gt(0).all()
assert set(data[STRATUM_COLUMN].unique()) == set(EXPECTED_CLUSTER_COUNTS)

cluster_sizes = (
    data.groupby([STRATUM_COLUMN, CLUSTER_COLUMN], observed=True)
    .agg(
        **{
            "비가중 N": (ID_COLUMN, "size"),
            "가중 사례 수": (WEIGHT_COLUMN, "sum"),
        }
    )
    .reset_index()
)
observed_cluster_counts = cluster_sizes.groupby(STRATUM_COLUMN)[
    CLUSTER_COLUMN
].nunique().to_dict()
assert observed_cluster_counts == EXPECTED_CLUSTER_COUNTS

cache_rows = []
for stratum in EXPECTED_CLUSTER_COUNTS:
    cache_path = AUTOENCODER_DIR / f"{stratum}_seed20260720.joblib"
    cached = joblib.load(cache_path)
    assert isinstance(cached, tuple) and len(cached) == 2
    embedding, validation_r2 = cached
    assert getattr(embedding, "ndim", None) == 2
    assert embedding.shape[1] == 16
    cache_rows.append(
        {
            "층": stratum,
            "캐시 파일": cache_path.name,
            "임베딩 행 수": embedding.shape[0],
            "임베딩 차원": embedding.shape[1],
            "검증 R²": validation_r2,
            "디코더 저장 여부": False,
            "새 재구성 성능 계산": "미수행",
        }
    )
cache_audit = pd.DataFrame(cache_rows)
cache_audit.to_csv(
    TABLES_DIR / "autoencoder_cache_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

print("입력 품질 및 공식 군집 수 확인 결과")
display(cluster_sizes.round(2))
print("오토인코더 캐시 점검 결과")
display(cache_audit.round(4))

In [ ]:
def parse_value_labels(value: object) -> dict[str, str]:
    """코드북의 JSON 값 라벨을 문자열 키 사전으로 바꾼다.

    Args:
        value: 코드북의 `integrated_value_labels` 값.

    Returns:
        코드값과 표시 라벨의 사전이다.
    """
    if pd.isna(value) or not str(value).strip():
        return {}
    parsed = json.loads(value)
    return {str(key): str(label).strip() for key, label in parsed.items()}


def canonical_code(value: object) -> str:
    """숫자 코드와 문자열 코드를 일관된 비교용 문자열로 바꾼다.

    Args:
        value: 코드값 또는 범주값.

    Returns:
        비교와 출력에 쓸 정규화한 코드 문자열이다.
    """
    if isinstance(value, (int, np.integer)):
        return str(int(value))
    if isinstance(value, (float, np.floating)) and float(value).is_integer():
        return str(int(value))
    return str(value).strip()


def is_non_substantive(label: str) -> bool:
    """결측·비해당·무응답 수준인지 판별한다.

    Args:
        label: 범주 수준의 표시 라벨.

    Returns:
        요약과 그림에서 제외할 수준이면 True이다.
    """
    keywords = ("결측", "비해당", "무응답", "모름", "모르겠", "없음")
    return any(keyword in label for keyword in keywords)


def weighted_mean(values: pd.Series, weights: pd.Series) -> tuple[float, int, float, int]:
    """결측을 쌍별 제외해 가중 평균과 분모 정보를 계산한다.

    Args:
        values: 평균 또는 비율을 계산할 수치형 값.
        weights: 같은 인덱스의 양수 가중치.

    Returns:
        가중 평균, 유효 비가중 N, 가중 분모, 결측 N의 튜플이다.
    """
    numeric = pd.to_numeric(values, errors="coerce")
    valid = numeric.notna() & weights.notna() & weights.gt(0)
    valid_n = int(valid.sum())
    missing_n = int((~numeric.notna()).sum())
    denominator = float(weights.loc[valid].sum())
    if denominator <= 0:
        return np.nan, valid_n, denominator, missing_n
    mean = float(np.average(numeric.loc[valid], weights=weights.loc[valid]))
    return mean, valid_n, denominator, missing_n


def prepare_metadata(
    codebook: pd.DataFrame, display_names: pd.DataFrame
) -> tuple[dict[str, dict[str, object]], dict[str, dict[str, str]]]:
    """코드북과 표시명 파일에서 변수 메타데이터를 만든다.

    Args:
        codebook: 통합 전처리 변수 코드북.
        display_names: 원변수의 짧은 한국어 표시명 파일.

    Returns:
        변수 메타데이터와 변수별 값 라벨 사전의 튜플이다.
    """
    required_columns = {"column_name", "display_name"}
    assert required_columns.issubset(display_names.columns)
    assert display_names["column_name"].is_unique
    display_name_map = display_names.set_index("column_name")[
        "display_name"
    ].to_dict()

    metadata = {}
    value_labels = {}
    for row in codebook.itertuples(index=False):
        metadata[row.column_name] = {
            "label": display_name_map.get(row.column_name, row.column_label),
            "codebook_label": row.column_label,
            "column_type": row.column_type,
        }
        value_labels[row.column_name] = parse_value_labels(
            row.integrated_value_labels
        )
    return metadata, value_labels


def is_total_spending_variable(
    column: str, metadata: dict[str, dict[str, object]]
) -> bool:
    """원변수 지출 총액을 lift 후보에서 제외할지 판별한다.

    Args:
        column: 확인할 원변수명.
        metadata: 코드북 기반 변수 메타데이터.

    Returns:
        출발 전 또는 여행 중 지출 총액 원변수이면 True이다.
    """
    label = str(metadata.get(column, {}).get("codebook_label", ""))
    spending_terms = ("지출비용", "지출 비용", "총액", "경비")
    return (
        column.startswith(("A7", "A8"))
        and any(term in label for term in spending_terms)
        and "1인지출" not in label
    )


def classify_features(
    frame: pd.DataFrame,
    metadata: dict[str, dict[str, object]],
    value_labels: dict[str, dict[str, str]],
) -> tuple[dict[str, str], pd.DataFrame]:
    """변수 유형을 정하고 분석 제외 사유를 기록한다.

    Args:
        frame: 군집 라벨과 후보 변수를 포함한 분석 자료.
        metadata: 코드북 기반 변수 메타데이터.
        value_labels: 코드북 기반 변수별 값 라벨.

    Returns:
        분석 변수의 유형 사전과 제외 사유 표의 튜플이다.
    """
    feature_types = {}
    exclusions = []
    key_columns = {ID_COLUMN, WEIGHT_COLUMN, STRATUM_COLUMN, CLUSTER_COLUMN}
    for column in frame.columns:
        if column in key_columns:
            exclusions.append({"변수": column, "제외 사유": "키·가중치·층·군집 열"})
            continue

        exclusion_reason = INTERPRETATION_EXCLUSIONS.get(column)
        if not exclusion_reason and is_total_spending_variable(
            column, metadata
        ):
            exclusion_reason = "1인당·1일당 지출 지표를 사용하므로 지출 총액은 제외"
        if exclusion_reason:
            exclusions.append({"변수": column, "제외 사유": exclusion_reason})
            continue

        non_missing_by_stratum = frame.groupby(
            STRATUM_COLUMN, observed=True
        )[column].apply(lambda values: values.notna().sum())
        if non_missing_by_stratum.eq(0).all():
            exclusions.append({"변수": column, "제외 사유": "전부 결측"})
            continue

        unique_by_stratum = frame.groupby(
            STRATUM_COLUMN, observed=True
        )[column].apply(lambda values: values.dropna().nunique())
        constant_strata = unique_by_stratum[unique_by_stratum.le(1)].index.tolist()
        if constant_strata:
            exclusions.append(
                {
                    "변수": column,
                    "제외 사유": "층 내 상수",
                    "해당 층": ", ".join(constant_strata),
                }
            )
            continue

        if column in DERIVED_TYPES:
            feature_types[column] = DERIVED_TYPES[column]
            continue

        codebook_info = metadata.get(column, {})
        labels = value_labels.get(column, {})
        if codebook_info.get("column_type") == "nominal" or labels:
            feature_types[column] = "범주형"
        else:
            feature_types[column] = "연속·계수형"

    for variable, levels in INTERPRETATION_EXCLUDED_LEVELS.items():
        if variable in frame.columns:
            exclusions.append(
                {
                    "변수": f"{variable}={', '.join(sorted(levels))}",
                    "제외 사유": "기타 활동 수준은 주관식 세부 확인이 필요해 제외",
                }
            )

    return feature_types, pd.DataFrame(exclusions)


def level_information(
    value: object, labels: dict[str, str]
) -> tuple[str, str, bool]:
    """범주값을 코드·라벨·실질 수준 여부로 바꾼다.

    Args:
        value: 원자료의 범주값.
        labels: 해당 변수의 값 라벨 사전.

    Returns:
        수준 코드, 수준 라벨, 실질 수준 여부의 튜플이다.
    """
    if pd.isna(value):
        return "결측·비해당", "결측·비해당", False
    code = canonical_code(value)
    label = labels.get(code, code)
    return code, label, not is_non_substantive(label)


metadata, value_label_map = prepare_metadata(codebook, display_names)
# A12 문항의 9는 해당사항 없음이다. 일부 연도 통합 코드북에서 라벨이
# 빠져 있어도 비실질 응답으로 일관되게 제외한다.
for question_number in range(1, 7):
    variable = f"A12_{question_number}"
    if variable in data.columns:
        value_label_map.setdefault(variable, {}).setdefault(
            "9", "해당사항 없음"
        )
assert set(DERIVED_DISPLAY_NAMES) == set(DERIVED_TYPES)
metadata.update(
    {
        variable: {
            "label": label,
            "codebook_label": label,
            "column_type": "derived",
        }
        for variable, label in DERIVED_DISPLAY_NAMES.items()
    }
)

## 유형별 lift 계산

연속·계수형은 군집 가중평균을 같은 층 가중평균으로 나눈다. 이진형과
범주형은 각 수준의 군집 내 가중 비율을 층 내 가중 비율로 나눈다. 결측·비해당
수준도 전체 결과에는 남기지만, 실질 특징 요약과 그림에서는 제외한다.

In [ ]:
def calculate_lift(
    frame: pd.DataFrame,
    feature_types: dict[str, str],
    metadata: dict[str, dict[str, object]],
    value_labels: dict[str, dict[str, str]],
) -> pd.DataFrame:
    """모든 후보 변수의 층·군집별 가중 lift를 계산한다.

    Args:
        frame: 공식 군집 라벨과 분석 변수를 포함한 자료.
        feature_types: 변수별 의미 유형.
        metadata: 코드북 기반 변수 메타데이터.
        value_labels: 코드북 기반 변수별 값 라벨 사전.

    Returns:
        변수와 수준별 전체 lift 결과표이다.
    """
    rows = []
    for stratum, stratum_frame in frame.groupby(STRATUM_COLUMN, observed=True):
        stratum_weights = pd.to_numeric(
            stratum_frame[WEIGHT_COLUMN], errors="coerce"
        )
        cluster_sizes = stratum_frame.groupby(
            CLUSTER_COLUMN, observed=True
        ).size()
        cluster_weights = stratum_weights.groupby(
            stratum_frame[CLUSTER_COLUMN], observed=True
        ).sum()
        clusters = cluster_sizes.index
        for variable, variable_type in feature_types.items():
            variable_label = metadata.get(variable, {}).get("label", variable)
            if variable_type == "연속·계수형":
                values = pd.to_numeric(stratum_frame[variable], errors="coerce")
                valid = values.notna()
                base_denominator = float(stratum_weights.loc[valid].sum())
                base_mean = (
                    float(
                        (values.loc[valid] * stratum_weights.loc[valid]).sum()
                        / base_denominator
                    )
                    if base_denominator > 0
                    else np.nan
                )
                cluster_key = stratum_frame[CLUSTER_COLUMN]
                denominators = (
                    stratum_weights.where(valid)
                    .groupby(cluster_key, observed=True)
                    .sum()
                    .reindex(clusters, fill_value=0.0)
                )
                weighted_sums = (
                    (values * stratum_weights)
                    .groupby(cluster_key, observed=True)
                    .sum()
                    .reindex(clusters, fill_value=0.0)
                )
                valid_n = valid.groupby(cluster_key, observed=True).sum().reindex(
                    clusters, fill_value=0
                )
                cluster_means = weighted_sums / denominators.replace(0, np.nan)
                reason = (
                    ""
                    if np.isfinite(base_mean) and base_mean > 0
                    else "층 기준 가중평균이 0 이하 또는 산출 불가"
                )
                for cluster in clusters:
                    cluster_mean = cluster_means.loc[cluster]
                    lift = (
                        cluster_mean / base_mean
                        if np.isfinite(base_mean) and base_mean > 0
                        else np.nan
                    )
                    rows.append({
                        "층": stratum, "군집": cluster,
                        "변수 유형": variable_type, "변수": variable,
                        "변수 라벨": variable_label,
                        "수준 코드": "연속형 평균", "수준 라벨": "가중평균",
                        "표시 특징": variable_label,
                        "군집 셀 비가중 N": int(cluster_sizes.loc[cluster]),
                        "유효 비가중 N": int(valid_n.loc[cluster]),
                        "수준 비가중 N": int(valid_n.loc[cluster]),
                        "결측 N": int(cluster_sizes.loc[cluster] - valid_n.loc[cluster]),
                        "가중 분모": float(denominators.loc[cluster]),
                        "군집값": cluster_mean, "층 기준값": base_mean,
                        "가중평균 차이": cluster_mean - base_mean,
                        "층 내 가중 비율": np.nan, "lift": lift,
                        "log2(lift)": np.log2(lift) if lift > 0 else np.nan,
                        "lift 미산출 사유": reason, "실질 수준 여부": True,
                    })
                continue

            labels = value_labels.get(variable, {})
            level_codes = stratum_frame[variable].map(
                lambda value: "결측·비해당"
                if pd.isna(value)
                else canonical_code(value)
            )
            raw_levels = pd.DataFrame(
                {"코드": level_codes, "원값": stratum_frame[variable]}
            ).drop_duplicates("코드")
            level_metadata = {
                row.코드: level_information(row.원값, labels)
                for row in raw_levels.itertuples(index=False)
            }
            level_order = pd.Index(level_metadata)
            base_weights = stratum_weights.groupby(level_codes, observed=True).sum()
            base_shares = base_weights / stratum_weights.sum()
            cluster_key = stratum_frame[CLUSTER_COLUMN]
            joint_weights = (
                pd.DataFrame(
                    {"군집": cluster_key, "수준": level_codes, "가중치": stratum_weights}
                )
                .groupby(["군집", "수준"], observed=True)["가중치"]
                .sum()
                .reindex(pd.MultiIndex.from_product([clusters, level_order]), fill_value=0.0)
            )
            joint_counts = (
                pd.DataFrame({"군집": cluster_key, "수준": level_codes})
                .groupby(["군집", "수준"], observed=True)
                .size()
                .reindex(pd.MultiIndex.from_product([clusters, level_order]), fill_value=0)
            )
            missing_counts = (
                level_codes.eq("결측·비해당")
                .groupby(cluster_key, observed=True)
                .sum()
                .reindex(clusters, fill_value=0)
            )
            for cluster, level_code in joint_weights.index:
                if level_code in INTERPRETATION_EXCLUDED_LEVELS.get(
                    variable, set()
                ):
                    continue
                level_label = level_metadata[level_code][1]
                substantive_level = level_metadata[level_code][2]
                cluster_share = float(
                    joint_weights.loc[(cluster, level_code)]
                    / cluster_weights.loc[cluster]
                )
                base_share = float(base_shares.loc[level_code])
                lift = cluster_share / base_share if base_share > 0 else np.nan
                display_level = (
                    level_label
                    if level_code == level_label
                    else f"{level_code}: {level_label}"
                )
                rows.append({
                    "층": stratum, "군집": cluster,
                    "변수 유형": variable_type, "변수": variable,
                    "변수 라벨": variable_label,
                    "수준 코드": level_code, "수준 라벨": level_label,
                    "표시 특징": f"{variable_label} — {display_level}",
                    "군집 셀 비가중 N": int(cluster_sizes.loc[cluster]),
                    "유효 비가중 N": int(cluster_sizes.loc[cluster]),
                    "수준 비가중 N": int(joint_counts.loc[(cluster, level_code)]),
                    "결측 N": int(missing_counts.loc[cluster]),
                    "가중 분모": float(cluster_weights.loc[cluster]),
                    "군집값": cluster_share, "층 기준값": base_share,
                    "가중평균 차이": cluster_share - base_share,
                    "층 내 가중 비율": base_share, "lift": lift,
                    "log2(lift)": np.log2(lift) if lift > 0 else np.nan,
                    "lift 미산출 사유": "",
                    "실질 수준 여부": substantive_level,
                })
    return pd.DataFrame(rows)


feature_types, exclusions = classify_features(data, metadata, value_label_map)
expected_derived = set(DERIVED_TYPES)
assert expected_derived.issubset(data.columns)
assert len(expected_derived) == 39

feature_type_table = pd.DataFrame(
    [
        {
            "변수": variable,
            "변수 라벨": metadata[variable]["label"],
            "변수 유형": kind,
        }
        for variable, kind in feature_types.items()
    ]
)
exclusions.to_csv(
    TABLES_DIR / "cluster_lift_exclusions.csv",
    index=False,
    encoding="utf-8-sig",
)
feature_type_table.to_csv(
    TABLES_DIR / "cluster_lift_feature_types.csv",
    index=False,
    encoding="utf-8-sig",
)

lift_all = calculate_lift(data, feature_types, metadata, value_label_map)
lift_all.to_csv(
    TABLES_DIR / "cluster_lift_all_variables.csv",
    index=False,
    encoding="utf-8-sig",
)

summary_candidates = lift_all.loc[
    lift_all["군집 셀 비가중 N"].ge(MIN_CLUSTER_N)
    & lift_all["실질 수준 여부"]
    & lift_all["log2(lift)"].notna()
    & (
        lift_all["변수 유형"].eq("연속·계수형")
        | lift_all["층 내 가중 비율"].ge(MIN_BASE_SHARE)
    )
].copy()
visual_summary_candidates = summary_candidates.loc[
    ~summary_candidates["변수"].isin(VISUAL_EXCLUDED_VARIABLES)
    & ~summary_candidates["변수"].str.startswith(
        VISUAL_EXCLUDED_PREFIXES
    )
    & ~summary_candidates["변수"].str.contains("_RANKW_", regex=False)
].copy()
summary = (
    visual_summary_candidates.assign(
        _absolute_log2=lambda frame: frame["log2(lift)"].abs()
    )
    .sort_values(
        [STRATUM_COLUMN, CLUSTER_COLUMN, "_absolute_log2"],
        ascending=[True, True, False],
    )
    .groupby([STRATUM_COLUMN, CLUSTER_COLUMN], observed=True)
    .head(TOP_FEATURES_PER_CLUSTER)
    .drop(columns="_absolute_log2")
)
summary.to_csv(
    TABLES_DIR / "cluster_lift_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

print(f"분석 변수: {len(feature_types):,}개")
print(f"제외 변수: {len(exclusions):,}개")
print(f"전체 lift 행: {len(lift_all):,}개")
print("군집별 대표 특징(교통·순위 중복 제외, 절대 log2(lift) 상위 12개)")
display(summary.head(20).round(4))

## 재현 검증과 시각화

기존 33개 수치형 lift는 원래 정의(이진형은 1의 가중비율, 그 밖은 가중평균)로
다시 계산해 저장본과 허용 오차 내에서 비교한다. 범주형과 연속형은 군집 결과를
가중 결합했을 때 층 기준값을 되찾는지도 검산한다.

In [ ]:
def calculate_legacy_numeric_lift(
    frame: pd.DataFrame, variables: list[str]
) -> pd.DataFrame:
    """기존 수치형 lift 정의를 그대로 다시 계산한다.

    Args:
        frame: 공식 군집 라벨링 자료.
        variables: 기존 lift 표에 있던 변수 목록.

    Returns:
        기존 형식과 맞춘 lift 재계산 결과표이다.
    """
    rows = []
    for stratum, stratum_frame in frame.groupby(STRATUM_COLUMN, observed=True):
        stratum_weights = pd.to_numeric(
            stratum_frame[WEIGHT_COLUMN], errors="coerce"
        )
        for variable in variables:
            base_mean, _, _, _ = weighted_mean(
                stratum_frame[variable], stratum_weights
            )
            for cluster, cluster_frame in stratum_frame.groupby(
                CLUSTER_COLUMN, observed=True
            ):
                cluster_mean, _, _, _ = weighted_mean(
                    cluster_frame[variable],
                    pd.to_numeric(
                        cluster_frame[WEIGHT_COLUMN], errors="coerce"
                    ),
                )
                rows.append({
                    "층": stratum,
                    "군집": cluster,
                    "변수": variable,
                    "재계산 lift": (
                        cluster_mean / base_mean if base_mean > 0 else np.nan
                    ),
                })
    return pd.DataFrame(rows)


def validate_weighted_reconstruction(lift_table: pd.DataFrame) -> pd.DataFrame:
    """군집 가중 결합으로 층 기준값이 복원되는지 검산한다.

    Args:
        lift_table: 전체 lift 결과표.

    Returns:
        변수·수준별 층 기준값 복원 오차 표이다.
    """
    rows = []
    for keys, subset in lift_table.groupby(
        ["층", "변수", "수준 코드"], dropna=False, observed=True
    ):
        valid = subset["군집값"].notna() & subset["가중 분모"].gt(0)
        denominator = subset.loc[valid, "가중 분모"].sum()
        reconstructed = np.nan
        if denominator > 0:
            reconstructed = float(np.average(
                subset.loc[valid, "군집값"],
                weights=subset.loc[valid, "가중 분모"],
            ))
        base_value = subset["층 기준값"].iloc[0]
        rows.append({
            "층": keys[0],
            "변수": keys[1],
            "수준 코드": keys[2],
            "층 기준값": base_value,
            "군집 가중 결합값": reconstructed,
            "절대 오차": abs(reconstructed - base_value),
        })
    return pd.DataFrame(rows)


prior_lift = pd.read_csv(PRIOR_LIFT_PATH)
legacy_variables = prior_lift["변수"].drop_duplicates().tolist()
legacy_recalculated = calculate_legacy_numeric_lift(data, legacy_variables)
legacy_check = prior_lift.merge(
    legacy_recalculated,
    on=["층", "군집", "변수"],
    how="outer",
    validate="one_to_one",
    indicator=True,
)
legacy_check["절대 오차"] = (
    legacy_check["lift"] - legacy_check["재계산 lift"]
).abs()
comparable = legacy_check["lift"].notna() & legacy_check["재계산 lift"].notna()
assert legacy_check["_merge"].eq("both").all()
assert legacy_check.loc[comparable, "절대 오차"].max() <= TOLERANCE
legacy_check.to_csv(
    TABLES_DIR / "cluster_lift_legacy_validation.csv",
    index=False,
    encoding="utf-8-sig",
)

reconstruction = validate_weighted_reconstruction(lift_all)
reconstruction.to_csv(
    TABLES_DIR / "cluster_lift_reconstruction_validation.csv",
    index=False,
    encoding="utf-8-sig",
)
assert reconstruction["절대 오차"].dropna().max() <= TOLERANCE

lift_cmap = LinearSegmentedColormap.from_list(
    "project_lift",
    [COLORS["primary"], COLORS["background"], COLORS["negative"]],
)
for stratum, stratum_summary in summary.groupby(STRATUM_COLUMN, observed=True):
    feature_order = (
        stratum_summary.assign(
            _absolute_log2=lambda frame: frame["log2(lift)"].abs()
        )
        .groupby("표시 특징", observed=True)["_absolute_log2"]
        .max()
        .sort_values(ascending=False)
        .index
    )
    heatmap_data = stratum_summary.pivot_table(
        index="표시 특징",
        columns=CLUSTER_COLUMN,
        values="lift",
        aggfunc="first",
    ).reindex(feature_order)
    figure, axis = plt.subplots(
        figsize=(8.5, max(5, 0.22 * len(heatmap_data)))
    )
    sns.heatmap(
        heatmap_data,
        cmap=lift_cmap,
        norm=TwoSlopeNorm(
            vmin=0,
            vcenter=1,
            vmax=max(2, heatmap_data.max().max()),
        ),
        annot=True,
        fmt=".2f",
        linewidths=0.4,
        linecolor=COLORS["background"],
        cbar_kws={"label": "같은 층 가중 비율·평균 대비 lift"},
        ax=axis,
    )
    axis.set_title(f"{stratum} 군집별 대표 특징 (WT_DOM 가중)")
    axis.set_xlabel("군집")
    axis.set_ylabel("변수 라벨 [원변수] — 수준")
    figure.tight_layout()
    figure.savefig(
        FIGURES_DIR / f"{stratum}_cluster_lift_heatmap.png",
        dpi=900,
        bbox_inches="tight",
    )
    plt.show()

print("기존 33개 수치형 lift 재현 검증")
display(legacy_check[["층", "군집", "변수", "lift", "재계산 lift", "절대 오차"]].head())
print(f"최대 절대 오차: {legacy_check.loc[comparable, '절대 오차'].max():.3e}")
print("군집 가중 결합의 층 기준값 복원 검증")
print(f"최대 절대 오차: {reconstruction['절대 오차'].dropna().max():.3e}")

## 해석 시 유의점

- `cluster_lift_all_variables.csv`는 결측·비해당·희소 수준을 포함한 전체 결과다.
- `cluster_lift_summary.csv`와 heatmap은 군집 셀 비가중 N 30 이상, 범주 수준의
  층 내 가중 비율 1% 이상을 만족하는 실질 수준 중 절대 log2(lift)가 큰 12개
  특징만 군집별로 제시한다.
- 일행 수는 원변수 `A6` 대신 파생 `party_size`만 사용한다.
- 소비는 총액 원변수와 중복 파생값을 제외하고 1인당·1인·1일당 지출 및 지출
  비중 지표만 사용한다.
- lift가 1보다 크면 해당 군집에서 같은 층 평균보다 과대표, 1보다 작으면
  과소대표다. 기준값이 0 이하인 연속·계수형은 lift를 산출하지 않는다.
- 수치형 지역·응답 코드는 범주형으로 계산했다. 따라서 코드 숫자의 평균에는
  의미를 부여하지 않는다.

In [ ]:
from src.notebook_sync import sync_script_from_notebook

print("최종 산출물")
for path in sorted(TABLES_DIR.glob("cluster_lift_*.csv")):
    print(f"- 표: {path.name}")
for path in sorted(FIGURES_DIR.glob("*_cluster_lift_heatmap.png")):
    print(f"- 그림: {path.name}")

sync_script_from_notebook(
    "notebooks/04_Clustering/260726_연고관광_전체변수_클러스터_lift.ipynb"
)